### Install packages (run once) 

In [1]:
%pip install boldigger3 pandas biopython openpyxl

Note: you may need to restart the kernel to use updated packages.


After installation restart kernel: Ctrl + Shift + P, write Kernel and choose Restart Kernel.

### Install Playwright browser (run once)

In [9]:
%pip install playwright

  Using cached pyee-13.0.1-py3-none-any.whl.metadata (3.0 kB)
Using cached pyee-13.0.1-py3-none-any.whl (15 kB)
  Attempting uninstall: pyee
    Found existing installation: pyee 14.0.0
    Uninstalling pyee-14.0.0:
      Successfully uninstalled pyee-14.0.0
Note: you may need to restart the kernel to use updated packages.


In [10]:
!python -m playwright install chromium

(node:8345) [DEP0169] DeprecationWarning: `url.parse()` behavior is not standardized and prone to errors that have security implications. Use the WHATWG URL API instead. CVEs are not issued for `url.parse()` vulnerabilities.
(Use `node --trace-deprecation ...` to show where the warning was created)
164.7 MiB [                    ] 0% 0.0s164.7 MiB [                    ] 0% 616.6s164.7 MiB [                    ] 0% 476.4s164.7 MiB [                    ] 0% 422.4s164.7 MiB [                    ] 0% 388.8s164.7 MiB [                    ] 0% 400.1s164.7 MiB [                    ] 0% 363.3s164.7 MiB [                    ] 0% 420.8s164.7 MiB [                    ] 0% 349.4s164.7 MiB [                    ] 0% 325.3s164.7 MiB [                    ] 0% 293.5s164.7 MiB [                    ] 0% 260.1s164.7 MiB [                    ] 0% 256.0s164.7 MiB [                    ] 0% 237.5s164.7 MiB [                    ] 0% 222.3s164.7 MiB [                    ] 0% 208.6s164.7 MiB [                   

### Create directories (use once)

In [ ]:
!mkdir -p ../data
!mkdir -p ../bold_db

### Download bold database (use once)

Enter this command in VScode terminal:
### boldigger3 download_db your_directory/bold_db

Change directory path by your desire, just remember to change the path of variable BOLD_DB_PATH = "../bold_db/BOLD_Public.11-Sep-2026.ddb".
It will ask you for your BOLD credentials and will download and unpack it (~2.39GB size). Just wait...

### Imports and configuration

In [ ]:
import subprocess
import pandas as pd
from Bio import SeqIO
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord
import logging
import sys

logging.basicConfig(level=logging.INFO)

# configurations

# Input excel file for convertion to fasta
INPUT_EXCEL = "../data/sequence_data.xlsx"
# Change column ID's as in excel file (you have two columns, one for sample ID and one for sequence)
ID_COLUMN = "Sample_ID"
SEQUENCE_COLUMN = "Sequence"
# Sheet name in excel file
SHEET = "Sheet1"
# Output fasta file for BOLD identification
CLEAN_FASTA = "../data/sequence_data.fasta"
# Path to the BOLD database. Change to your db name that has been downloaded from BOLD.
BOLD_DB_PATH = "../bold_db/BOLD_Public.11-Sep-2026.ddb"



### Define functions

In [ ]:
def excel_to_fasta(excel_path, fasta_path, id_col, seq_col, sheet_name=0):

    try:
        df = pd.read_excel(excel_path, sheet_name=sheet_name)

        records = []

        for index, row in df.iterrows():
            seq_id = str(row[id_col])
            sequence = str(row[seq_col])

            if not seq_id or not sequence:
                logging.warning(f"Skipping row {index + 2} in {excel_path}: Missing ID or sequence.")
                continue
            record = SeqRecord(Seq(sequence), id=seq_id, description="")
            records.append(record)

        if records:
            SeqIO.write(records, fasta_path, "fasta")
            logging.info(f"Successfully wrote {len(records)} sequences to {fasta_path}.")
        else:
            logging.warning("No valid sequences found to write.")
        
    except FileNotFoundError:
        logging.error(f"Error: The file {excel_path} was not found.")
    except KeyError as e:
        logging.error(f"Error: Column {e} not found in the Excel file. Please check your column names.")
    except Exception as e:
        logging.error(f"An unexpected error occurred: {e}")

   

### Activate excel_to_fasta function if you have such an excel

In [ ]:
excel_to_fasta(INPUT_EXCEL, CLEAN_FASTA, ID_COLUMN, SEQUENCE_COLUMN, SHEET)

### There are two ways to activate BOLD identification:
1). First one using bash command, much simplier. --db 1 is public database, --mode 2 is more reliable and faster identification. To see more of types to run you can use boldigger3 --help command or search for boldigger documentation.

In [ ]:
!boldigger3 identify ../data/sequence_data.fasta ../bold_db/BOLD_Public.11-Sep-2026.ddb --db 1 --mode 2

### Second way is to use the script below

In [ ]:
command = [
    "boldigger3", "identify",
    CLEAN_FASTA,
    BOLD_DB_PATH,
    "--db", "2",
    "--mode", "2"
]

try:
    result = subprocess.run(command, check=True, capture_output=True, text=True)
    logging.info("BOLD identification completed successfully.")
    logging.info(result.stdout)
except subprocess.CalledProcessError as e:
    logging.error(f"Error during BOLD identification: {e}.")
    logging.error(e.stderr)
    sys.exit(1)

### Results will be stored at data/boldigger3_data